## Process Historical S&P 500 Constituents by Rebalance Cycle
* Divide the historical timeline into rebalance cycles.
* Filter eligible S&P500 constituents as of each rebalance reference date
* Store the constituent set for each cycle for later momentum calculation.
* Method: [S&P500 Momentum Index Methodology](https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-momentum-indices.pdf)

In [9]:
import pandas as pd
import numpy as np

In [13]:
sp_membership = pd.read_parquet("../data/raw/sp500_membership.parquet")
sp500 = pd.read_parquet("../data/raw/sp500.parquet")

In [11]:
sp_membership

,permno,start,ending
0,10006,1957-03-01,1984-07-18
1,10030,1957-03-01,1969-01-08
2,10049,1925-12-31,1932-10-01
3,10057,1957-03-01,1992-07-02
4,10078,1992-08-20,2010-01-28
...,...,...,...
2059,93159,2012-07-31,2016-03-29
2060,93246,2021-03-22,2024-12-31
2061,93422,2010-07-01,2015-06-30
2062,93429,2017-03-01,2024-12-31


### Get all rebalance reference date & effective date (1958 - 2024)
According to [S&P500 Momentum Index Methodology](https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-momentum-indices.pdf):
> [S&P 500 Momentum] indices rebalance semi-annually, effective after the close on the third Friday of March and September, with a reference date of the last business day of February and August

In [21]:
# effective date
def third_friday(year, month):
    month_days = pd.date_range(
        start=pd.Timestamp(year, month, 1),
        end=pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0),
        freq="D"
    )

    fridays = month_days[month_days.weekday == 4]
    return fridays[2]

# reference date
def last_market_date(trade_dates, year, month):
    trade_dates = pd.DatetimeIndex(trade_dates)

    available = trade_dates[(trade_dates.year == year) &(trade_dates.month == month)]

    if len(available) == 0:
        return pd.NaT

    return available.max()

# actual holding date
def next_market_date(trade_dates, date):
    trade_dates = pd.DatetimeIndex(trade_dates).sort_values()
    later_dates = trade_dates[trade_dates > date]

    if len(later_dates) == 0:
        return pd.NaT

    return later_dates[0]        
        

In [22]:
def get_rebalance_cycles(trade_dates, start_year, end_year):
    records = []
    for year in range(start_year, end_year + 1):
        march_effective = third_friday(year, 3)
        feb_ref = last_market_date(trade_dates, year, 2)
        
        records.append({
            "cycle_id": f"{year}_03",
            "reference_date": feb_ref,
            "effective_date": march_effective,
            "holding_start": next_market_date(trade_dates, march_effective)
        })
        
        sept_effective = third_friday(year, 9)
        aug_ref = last_market_date(trade_dates, year, 8)
        
        records.append({
            "cycle_id": f"{year}_09",
            "reference_date": aug_ref,
            "effective_date": sept_effective,
            "holding_start": next_market_date(trade_dates, sept_effective)
        })
    schedule = pd.DataFrame(records)
    schedule["holding_end"] = schedule["holding_start"].shift(-1).apply(lambda x: trade_dates[trade_dates < x].max())
    return schedule  

In [24]:
trade_days = sp500["date"].unique()

rebalance_cycles = get_rebalance_cycles(trade_days, 1958, 2024)
rebalance_cycles

,cycle_id,reference_date,effective_date,holding_start,holding_end
0,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
1,1958_09,1958-08-29,1958-09-19,1958-09-22,1959-03-20
2,1959_03,1959-02-27,1959-03-20,1959-03-23,1959-09-18
3,1959_09,1959-08-31,1959-09-18,1959-09-21,1960-03-18
4,1960_03,1960-02-29,1960-03-18,1960-03-21,1960-09-16
...,...,...,...,...,...
129,2022_09,2022-08-31,2022-09-16,2022-09-19,2023-03-17
130,2023_03,2023-02-28,2023-03-17,2023-03-20,2023-09-15
131,2023_09,2023-08-31,2023-09-15,2023-09-18,2024-03-15
132,2024_03,2024-02-29,2024-03-15,2024-03-18,2024-09-20


### Get S&P500 Constituents for each rebalance cycle 
* Filter based on ...? on reference date